# Notebook 05 — Calibration review (auto-disable triage)

Reads `calibration_loss_history`; plots per-model drift
trajectories with the rolling baseline + z-threshold bands.
Surfaces models that are trending toward auto-disable so the
operator can investigate BEFORE the suppression fires.

_Spec_: `docs/ROUND_13_CALIBRATION_AND_RESEARCH.md` § 3.5


## Setup


In [ ]:
%run 00_data_loader.ipynb


## Per-model loss trajectories


In [ ]:
import matplotlib.pyplot as plt, pandas as pd

df = con.execute('''
    SELECT model, strategy_class, measured_at,
           brier_score, mape, log_loss, n_decisions
    FROM calibration_loss_history
    ORDER BY measured_at
''').fetchdf()

if df.empty:
    print('NO DATA — the calibration daemon has not run yet. Start polymarket-calibration.service.')
else:
    for model, sub in df.groupby('model'):
        loss = sub.brier_score.fillna(sub.mape).fillna(sub.log_loss)
        if loss.dropna().empty:
            continue
        plt.figure(figsize=(10, 3))
        plt.plot(sub.measured_at, loss, marker='.')
        plt.title(f'{model} — calibration loss')
        plt.xlabel('day'); plt.ylabel('loss'); plt.grid(alpha=0.3)
        plt.show()


## Currently-disabled models


In [ ]:
import asyncio, nest_asyncio
nest_asyncio.apply()
try:
    from src.calibration import get_auto_disabler
    rows = asyncio.run(get_auto_disabler().list_disabled())
    if not rows:
        print('No models currently disabled.')
    else:
        display(pd.DataFrame(rows))
except Exception as e:
    print(f'Auto-disabler unavailable: {e}')
